In [20]:
!pip install -q transformers==4.44.2 sentencepiece accelerate

### Import Libraries

In [21]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import pipeline

### Load dataset

In [22]:
df = pd.read_csv("/content/amazon_reviews_clustered.csv")

In [23]:
df.head()

,id,name,brand,categories,asins,reviews.rating,reviews.title,reviews.text,reviews.username,reviews.date,review_length,cluster_text,Cluster,Meta_Category
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Kindle,This product so far has not disappointed. My c...,Adapter,2017-01-13 00:00:00+00:00,27,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,very fast,great for beginner or experienced person. Boug...,truman,2017-01-13 00:00:00+00:00,14,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,DaveZ,2017-01-13 00:00:00+00:00,26,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,4.0,Good!!!,I've had my Fire HD 8 two weeks now and I love...,Shacks,2017-01-13 00:00:00+00:00,117,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,explore42,2017-01-12 00:00:00+00:00,117,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders


### Find the Top 3 Products in Each Category

In [24]:
product_stats = (
    df.groupby(["Meta_Category", "name"])
      .agg(
          avg_rating=("reviews.rating", "mean"),
          review_count=("reviews.rating", "count")
      )
      .reset_index()
)

### Keep products with enough reviews

In [25]:
product_stats = product_stats[
    product_stats["review_count"] >= 20
]

### Select Top 3 Products

In [26]:
top_products = (
    product_stats
    .sort_values(
        ["Meta_Category", "avg_rating"],
        ascending=[True, False]
    )
    .groupby("Meta_Category")
    .head(3)
)

top_products

,Meta_Category,name,avg_rating,review_count
0,Batteries & Household Essentials,AmazonBasics AA Performance Alkaline Batteries...,4.414875,3442
1,Batteries & Household Essentials,AmazonBasics AAA Performance Alkaline Batterie...,4.397141,7554
5,Electronics & Streaming Devices,Unknown Product,4.707278,5056
16,Smart Home & Alexa Devices,"Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",4.773438,128
8,Smart Home & Alexa Devices,Amazon - Echo Plus w/ Built-In Hub - Silver,4.748727,589
7,Smart Home & Alexa Devices,Amazon - Amazon Tap Portable Bluetooth and Wi-...,4.729560,318
68,Tablets & eReaders,Amazon Kindle Paperwhite - eBook reader - 4 GB...,4.755038,3176
123,Tablets & eReaders,"Kindle Voyage E-reader, 6 High-Resolution Disp...",4.743103,580
61,Tablets & eReaders,Amazon Fire Hd 8 8in Tablet 16gb Black B018szt...,4.740741,135


### Collect Reviews of Those Products

In [27]:
selected_reviews = df.merge(
    top_products[
        ["Meta_Category","name"]
    ],
    on=["Meta_Category","name"]
)

In [28]:
print("Total reviews:", len(selected_reviews))

Total reviews: 20978


### Split Reviews into Sentences

In [29]:
def split_sentences(text):

    text = str(text)

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if 15 < len(s.strip()) < 300
    ]

### TF-IDF Sentence Selection

In [30]:
def top_representative_sentences(texts, n=40):

    all_sentences = []

    for text in texts:

        all_sentences.extend(
            split_sentences(text)
        )

    if len(all_sentences)==0:
        return []

    # Remove duplicate sentences

    all_sentences = list(
        dict.fromkeys(all_sentences)
    )

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=2000
    )

    X = vectorizer.fit_transform(all_sentences)

    scores = np.asarray(
        X.sum(axis=1)
    ).ravel()

    top_idx = scores.argsort()[::-1][:n]

    return [
        all_sentences[i]
        for i in sorted(top_idx)
    ]

In [31]:
print(selected_reviews.head())

                     id                                               name  \
0  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
1  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
2  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
3  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
4  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   

    brand                                         categories       asins  \
0  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
1  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
2  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
3  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
4  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   

   reviews.rating          reviews.title  \
0             3.0      I like 

In [32]:
category = "Tablets & eReaders"

texts = selected_reviews.loc[
    selected_reviews["Meta_Category"] == category,
    "reviews.text"
]

### Create the input text

In [38]:
selected_sentences = top_representative_sentences(texts, n=40)

### Split into chunks

In [40]:
chunks = chunk_sentences(selected_sentences, chunk_size=10)

print("Number of chunks:", len(chunks))

Number of chunks: 4


### Load DistilBART

In [41]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=-1
)

### Summarize each chunk

In [42]:
chunk_summaries = []

for chunk in chunks:

    text = " ".join(chunk)

    summary = summarizer(
        text,
        max_length=120,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]

    chunk_summaries.append(summary)

In [45]:
for i, s in enumerate(chunk_summaries, 1):
    print(f"Chunk {i} Summary:\n{s}\n")

Chunk 1 Summary:
 The e-ink screen almost feels like a page of a paperback when you swipe, and in my personal preference, I like that . The battery is rated to last 6-8 weeks on average between the various tech sites .

Chunk 2 Summary:
 E-ink display allows you to also read outside in direct sunlight, perfect for the upcoming summer months . Navigation is a bit cumbersome since there are no physical buttons, the "BACK" button in particular is missed .

Chunk 3 Summary:
 The Kindle Paperwhite is reliable, holds a charge for weeks, is lightweight and easy to hold, allows you to read in bed at night without eye strain or bothering anyone (because of the backlit surface) It does have a few less LEDs for backlighting, no flush glass cover (but that is what broke) No auto-adjusting light (that never quite worked anyway), squeezy sides, and a tiny bit heavier/bigger than the Voyage .

Chunk 4 Summary:
 This is the only Kindle to feature the auto-brightness feature (not even the Oasis has it-

### Combine all chunk summaries

In [43]:
combined_summary = " ".join(chunk_summaries)

### Generate the final summary

In [44]:
final_summary = summarizer(
    combined_summary,
    max_length=180,
    min_length=80,
    do_sample=False
)[0]["summary_text"]

print(final_summary)

 The Kindle Paperwhite is reliable, holds a charge for weeks, is lightweight and easy to hold . The e-ink screen almost feels like a page of a paperback when you swipe, and in my personal preference, I like that . E-ink display allows you to also read outside in direct sunlight, perfect for the upcoming summer months . Navigation is a bit cumbersome since there are no physical buttons, the "BACK" button in particular is missed .


### Save the Result

In [46]:
results = pd.DataFrame({
    "Meta_Category": [category],
    "DistilBART_Summary": [final_summary]
})

results

,Meta_Category,DistilBART_Summary
0,Tablets & eReaders,"The Kindle Paperwhite is reliable, holds a ch..."


### Summarize all 4 categories

In [47]:
distilbart_results = []

for category in selected_reviews["Meta_Category"].unique():

    texts = selected_reviews.loc[
        selected_reviews["Meta_Category"] == category,
        "reviews.text"
    ]

    selected_sentences = top_representative_sentences(texts, n=40)

    chunks = chunk_sentences(selected_sentences, chunk_size=10)

    chunk_summaries = []

    for chunk in chunks:
        text = " ".join(chunk)

        summary = summarizer(
            text,
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]

        chunk_summaries.append(summary)

    combined_summary = " ".join(chunk_summaries)

    final_summary = summarizer(
        combined_summary,
        max_length=180,
        min_length=80,
        do_sample=False
    )[0]["summary_text"]

    distilbart_results.append({
        "Meta_Category": category,
        "DistilBART_Summary": final_summary
    })

distilbart_results = pd.DataFrame(distilbart_results)

distilbart_results

,Meta_Category,DistilBART_Summary
0,Tablets & eReaders,"The Kindle Paperwhite is reliable, holds a ch..."
1,Smart Home & Alexa Devices,Amazon Echo is a great device to use for list...
2,Electronics & Streaming Devices,4K compatible and microSD card slot makes thi...
3,Batteries & Household Essentials,AmazonBasics batteries hold up really well in...


### Save it as csv

In [48]:
distilbart_results.to_csv("distilbart_summaries.csv", index=False)

In [49]:
summarizer.model.save_pretrained("distilbart_model")
summarizer.tokenizer.save_pretrained("distilbart_model")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


('distilbart_model/tokenizer_config.json',
 'distilbart_model/special_tokens_map.json',
 'distilbart_model/vocab.json',
 'distilbart_model/merges.txt',
 'distilbart_model/added_tokens.json',
 'distilbart_model/tokenizer.json')

In [50]:
distilbart_results.to_pickle("distilbart_summaries.pkl")